In [ ]:
import torch
import torch.nn as nn

eng_vocab = {
    "<PAD>":0,
    "<SOS>":1,
    "<EOS>":2,
    "i":3,
    "love":4,
    "india":5
}

hin_vocab = {
    "<PAD>":0,
    "<SOS>":1,
    "<EOS>":2,
    "मैं":3,
    "भारत":4,
    "से":5,
    "प्यार":6,
    "करता":7,
    "हूँ":8
}

SRC_VOCAB = len(eng_vocab)
TGT_VOCAB = len(hin_vocab)

EMBED_SIZE = 32
HEADS = 4
LAYERS = 2
FFN = 64
MAX_LEN = 10

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2)
            * (-torch.log(torch.tensor(10000.0)) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class Translator(nn.Module):
    def __init__(self):
        super().__init__()

        self.src_embedding = nn.Embedding(
            SRC_VOCAB,
            EMBED_SIZE
        )

        self.tgt_embedding = nn.Embedding(
            TGT_VOCAB,
            EMBED_SIZE
        )

        self.pos = PositionalEncoding(
            EMBED_SIZE,
            MAX_LEN
        )

        self.transformer = nn.Transformer(
            d_model=EMBED_SIZE,
            nhead=HEADS,
            num_encoder_layers=LAYERS,
            num_decoder_layers=LAYERS,
            dim_feedforward=FFN,
            batch_first=True
        )

        self.fc = nn.Linear(
            EMBED_SIZE,
            TGT_VOCAB
        )

    def forward(self, src, tgt):
        src = self.pos(
            self.src_embedding(src)
        )

        tgt = self.pos(
            self.tgt_embedding(tgt)
        )

        output = self.transformer(src, tgt)
        return self.fc(output)

In [ ]:
src = torch.tensor([
    [
        eng_vocab["i"],
        eng_vocab["love"],
        eng_vocab["india"],
        eng_vocab["<EOS>"]
    ]
]).to(device)

In [ ]:
tgt_input = torch.tensor([
    [
        hin_vocab["<SOS>"],
        hin_vocab["मैं"],
        hin_vocab["भारत"],
        hin_vocab["से"],
        hin_vocab["प्यार"],
        hin_vocab["करता"]
    ]
]).to(device)

In [ ]:
target = torch.tensor([
    [
        hin_vocab["मैं"],
        hin_vocab["भारत"],
        hin_vocab["से"],
        hin_vocab["प्यार"],
        hin_vocab["करता"],
        hin_vocab["हूँ"]
    ]
]).to(device)

In [ ]:
model = Translator().to(device)

In [ ]:
output = model(src, tgt_input)

print(output.shape)

torch.Size([1, 6, 9])


In [ ]:
criterion = nn.CrossEntropyLoss()

loss = criterion(
    output.reshape(-1, TGT_VOCAB),
    target.reshape(-1)
)

print(loss)

tensor(2.2783, grad_fn=<NllLossBackward0>)


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
for epoch in range(301):

    optimizer.zero_grad()
    output = model(src, tgt_input)

    loss = criterion(
        output.reshape(-1, TGT_VOCAB),
        target.reshape(-1)
    )

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} Loss: {loss.item():.4f}")

Epoch 0 Loss: 2.3600
Epoch 50 Loss: 0.3321
Epoch 100 Loss: 0.1021
Epoch 150 Loss: 0.0499
Epoch 200 Loss: 0.0298
Epoch 250 Loss: 0.0241
Epoch 300 Loss: 0.0179


In [ ]:
with torch.no_grad():
    prediction = model(src, tgt_input)
    predicted_ids = prediction.argmax(dim=-1)

    print(predicted_ids)

tensor([[3, 4, 5, 6, 7, 8]])


In [ ]:
hin_vocab

{'<PAD>': 0,
 '<SOS>': 1,
 '<EOS>': 2,
 'मैं': 3,
 'भारत': 4,
 'से': 5,
 'प्यार': 6,
 'करता': 7,
 'हूँ': 8}